In [1]:

from pathlib import Path
import json
import sys
import os
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

direktori_aktif = Path.cwd()

if direktori_aktif.name.lower() == "notebooks":
    direktori_project = direktori_aktif.parent
else:
    direktori_project = direktori_aktif

direktori_src = direktori_project / "src"
direktori_outputs = direktori_project / "reports" / "outputs"
direktori_corrections = direktori_project / "data" / "corrections"
direktori_examples = direktori_project / "examples"

for direktori in [direktori_src, direktori_outputs, direktori_corrections, direktori_examples]:
    direktori.mkdir(parents=True, exist_ok=True)

if str(direktori_src) not in sys.path:
    sys.path.insert(0, str(direktori_src))

print("Direktori aktif notebook:", direktori_aktif)
print("Direktori project:", direktori_project)
print("Folder src:", direktori_src)
print("Folder outputs:", direktori_outputs)
print("Folder corrections:", direktori_corrections)
print("Folder examples:", direktori_examples)


Direktori aktif notebook: C:\Users\ASUS\PHISHING\notebooks
Direktori project: C:\Users\ASUS\PHISHING
Folder src: C:\Users\ASUS\PHISHING\src
Folder outputs: C:\Users\ASUS\PHISHING\reports\outputs
Folder corrections: C:\Users\ASUS\PHISHING\data\corrections
Folder examples: C:\Users\ASUS\PHISHING\examples


## Validasi File Wajib

In [2]:

file_wajib = {
    "phishrisk_engine_v3.py": direktori_src / "phishrisk_engine_v3.py",
    "url_intelligence.py": direktori_src / "url_intelligence.py",
    "file_static_analyzer.py": direktori_src / "file_static_analyzer.py",
    "run_phishrisk.py": direktori_src / "run_phishrisk.py",
    "model_terbaik_intelligence_v2.pkl": direktori_project / "models" / "model_terbaik_intelligence_v2.pkl",
    "daftar_fitur_intelligence_v2.json": direktori_outputs / "daftar_fitur_intelligence_v2.json",
}

data_validasi_awal = []

for nama_file, lokasi in file_wajib.items():
    data_validasi_awal.append(
        {
            "nama_file": nama_file,
            "lokasi": str(lokasi),
            "tersedia": lokasi.exists(),
            "ukuran_kb": round(lokasi.stat().st_size / 1024, 2) if lokasi.exists() else 0,
        }
    )

data_validasi_awal = pd.DataFrame(data_validasi_awal)
display(data_validasi_awal)

if not data_validasi_awal["tersedia"].all():
    raise FileNotFoundError("Ada file wajib yang belum tersedia. Cek hasil tabel validasi.")

print("Semua file wajib tersedia.")


,nama_file,lokasi,tersedia,ukuran_kb
0,phishrisk_engine_v3.py,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v3.py,True,17.66
1,url_intelligence.py,C:\Users\ASUS\PHISHING\src\url_intelligence.py,True,15.60
2,file_static_analyzer.py,C:\Users\ASUS\PHISHING\src\file_static_analyze...,True,17.95
3,run_phishrisk.py,C:\Users\ASUS\PHISHING\src\run_phishrisk.py,True,7.17
4,model_terbaik_intelligence_v2.pkl,C:\Users\ASUS\PHISHING\models\model_terbaik_in...,True,49114.43
5,daftar_fitur_intelligence_v2.json,C:\Users\ASUS\PHISHING\reports\outputs\daftar_...,True,1.02


Semua file wajib tersedia.


## Membuat Modul AI Safety Guard

In [3]:

isi_ai_safety_guard = r'''from __future__ import annotations

import re
from typing import Any, Dict, Iterable, Tuple


KATA_BERBAHAYA = [
    "buat phishing",
    "membuat phishing",
    "phishing kit",
    "clone login",
    "curi password",
    "steal password",
    "credential theft",
    "ambil kredensial",
    "bypass antivirus",
    "evade antivirus",
    "payload malware",
    "buat malware",
    "trojan",
    "ransomware",
    "keylogger",
    "exploit",
    "sql injection",
    "bruteforce",
    "brute force",
    "ddos",
    "menyerang",
    "hack akun",
    "ambil cookie",
    "session hijacking",
    "backdoor",
]


KATA_AMAN = [
    "deteksi",
    "analisis",
    "edukasi",
    "rekomendasi",
    "laporan",
    "defensif",
    "pencegahan",
    "antisipasi",
    "periksa",
    "jelaskan",
]


def bersihkan_teks(teks: Any, batas_karakter: int = 4000) -> str:
    """Membersihkan teks agar aman dipakai sebagai konteks AI."""
    if teks is None:
        return ""

    teks_bersih = str(teks)
    teks_bersih = teks_bersih.replace("\x00", " ")
    teks_bersih = re.sub(r"\s+", " ", teks_bersih).strip()

    if len(teks_bersih) > batas_karakter:
        teks_bersih = teks_bersih[:batas_karakter] + "..."

    return teks_bersih


def deteksi_permintaan_berbahaya(teks: Any) -> Tuple[bool, str]:
    """Mendeteksi permintaan yang mengarah ke penyalahgunaan."""
    teks_bersih = bersihkan_teks(teks, batas_karakter=2000).lower()

    for kata in KATA_BERBAHAYA:
        if kata in teks_bersih:
            return True, f"Permintaan mengandung pola berisiko: {kata}"

    return False, ""


def format_nilai_ringkas(nilai: Any) -> str:
    """Mengubah nilai menjadi teks pendek."""
    if nilai is None:
        return ""

    if isinstance(nilai, float):
        return f"{nilai:.2f}"

    return bersihkan_teks(nilai, batas_karakter=500)


def amankan_konteks_dict(data: Dict[str, Any], kolom_dipakai: Iterable[str] | None = None) -> Dict[str, str]:
    """Mengambil kolom penting dari dictionary hasil engine."""
    if data is None:
        return {}

    if kolom_dipakai is None:
        kolom_dipakai = data.keys()

    hasil: Dict[str, str] = {}

    for kolom in kolom_dipakai:
        if kolom in data:
            hasil[kolom] = format_nilai_ringkas(data.get(kolom))

    return hasil


def potong_daftar_teks(daftar_teks: Iterable[Any], batas_item: int = 20, batas_karakter_item: int = 400) -> list[str]:
    """Membatasi jumlah item agar konteks tetap ringan."""
    hasil = []

    for item in list(daftar_teks)[:batas_item]:
        hasil.append(bersihkan_teks(item, batas_karakter=batas_karakter_item))

    return hasil'''

lokasi_ai_safety_guard = direktori_src / "ai_safety_guard.py"
lokasi_ai_safety_guard.write_text(isi_ai_safety_guard, encoding="utf-8")

print("Modul AI Safety Guard berhasil dibuat:")
print(lokasi_ai_safety_guard)


Modul AI Safety Guard berhasil dibuat:
C:\Users\ASUS\PHISHING\src\ai_safety_guard.py


## Membuat Modul AI Explainer

Modul ini memiliki dua mode:

1. **Mode AI eksternal** jika `OPENAI_API_KEY` tersedia.
2. **Mode fallback lokal** jika API key belum ada.

Fallback lokal tetap bisa jalan, jadi notebook tidak macet walaupun belum memakai API AI.

In [4]:

isi_ai_explainer = r'''from __future__ import annotations

import json
import os
from typing import Any, Dict, Iterable, Optional

import pandas as pd

from ai_safety_guard import (
    amankan_konteks_dict,
    bersihkan_teks,
    deteksi_permintaan_berbahaya,
    potong_daftar_teks,
)


KOLOM_URL_PENTING = [
    "url",
    "domain",
    "label_model",
    "skor_model",
    "skor_final",
    "kategori_risiko",
    "hasil_akhir",
    "intelligence_status",
    "official_brand",
    "brand_detected",
    "suspicious_keywords",
    "lookalike_brand",
    "lookalike_score",
    "rekomendasi",
]

KOLOM_FILE_PENTING = [
    "nama_file",
    "ekstensi",
    "ukuran_kb",
    "jumlah_url",
    "jumlah_url_berisiko_v3",
    "jumlah_url_perlu_tinjauan_v3",
    "jumlah_kata_mencurigakan",
    "kata_mencurigakan",
    "skor_final_file_v3",
    "kategori_final_file_v3",
    "hasil_akhir_file_v3",
    "alasan_file",
    "rekomendasi_final_file_v3",
]


class PhishRiskAIExplainer:
    """Lapisan AI untuk menjelaskan hasil PhishRisk tanpa mengganti keputusan engine."""

    def __init__(
        self,
        aktifkan_ai: Optional[bool] = None,
        model_ai: Optional[str] = None,
        batas_karakter: int = 3500,
    ) -> None:
        self.model_ai = model_ai or os.getenv("PHISHRISK_AI_MODEL", "gpt-5.5-mini")
        self.batas_karakter = batas_karakter

        if aktifkan_ai is None:
            aktifkan_ai = bool(os.getenv("OPENAI_API_KEY"))

        self.aktifkan_ai = aktifkan_ai
        self._client = None

        if self.aktifkan_ai:
            self._client = self._siapkan_client()

    def _siapkan_client(self) -> Any:
        """Menyiapkan client AI jika paket dan API key tersedia."""
        try:
            from openai import OpenAI

            return OpenAI()
        except Exception:
            self.aktifkan_ai = False
            return None

    def _panggil_ai(self, prompt: str) -> str:
        """Memanggil AI jika aktif. Jika gagal, gunakan fallback lokal."""
        if not self.aktifkan_ai or self._client is None:
            return ""

        try:
            response = self._client.responses.create(
                model=self.model_ai,
                input=prompt,
            )
            return bersihkan_teks(getattr(response, "output_text", ""), batas_karakter=self.batas_karakter)
        except Exception:
            self.aktifkan_ai = False
            return ""

    def _prompt_sistem(self) -> str:
        return (
            "Anda adalah asisten keamanan defensif untuk PhishRisk. "
            "Tugas Anda hanya menjelaskan hasil deteksi URL/file secara aman, ringkas, dan mudah dipahami. "
            "Jangan memberi langkah menyerang, membuat phishing, membuat malware, bypass keamanan, atau mencuri data. "
            "Jangan mengganti keputusan engine. Gunakan hasil engine sebagai sumber utama. "
            "Bahasa wajib Indonesia yang sederhana."
        )

    def _fallback_url(self, hasil_url: Dict[str, Any]) -> Dict[str, str]:
        data = amankan_konteks_dict(hasil_url, KOLOM_URL_PENTING)
        hasil = data.get("hasil_akhir", "-")
        kategori = data.get("kategori_risiko", "-")
        status = data.get("intelligence_status", "-")
        brand = data.get("brand_detected") or data.get("official_brand") or data.get("lookalike_brand") or "-"
        kata = data.get("suspicious_keywords", "")
        rekomendasi_engine = data.get("rekomendasi", "")

        alasan = []

        if status == "resmi_terlihat_aman":
            alasan.append("Domain cocok dengan daftar pembanding resmi.")
        if "tiruan_brand" in status:
            alasan.append("Alamat terlihat memakai nama brand, tetapi bukan domain resmi.")
        if "domain_mirip_brand" in status:
            alasan.append("Domain terlihat mirip dengan brand resmi.")
        if kata:
            alasan.append(f"Ada kata yang perlu diwaspadai: {kata}.")
        if data.get("skor_final"):
            alasan.append(f"Skor akhir sistem adalah {data.get('skor_final')}.")

        if not alasan:
            alasan.append("Sistem tidak menemukan sinyal utama yang cukup kuat dari data yang tersedia.")

        if hasil == "Berisiko":
            tindakan = "Jangan buka link, jangan login, jangan isi data pribadi, dan cek domain resmi dari sumber tepercaya."
        elif hasil == "Perlu Tinjauan":
            tindakan = "Cek ulang domain dari sumber resmi. Jangan gunakan link dari pesan asing."
        else:
            tindakan = "Tetap cek ulang alamat sebelum login atau transaksi."

        return {
            "mode": "fallback_lokal",
            "ringkasan": f"Hasil engine: {hasil} dengan kategori {kategori}.",
            "alasan_sederhana": " ".join(alasan),
            "rekomendasi": rekomendasi_engine or tindakan,
            "brand_terkait": brand,
            "batasan": "Penjelasan ini membantu membaca hasil engine, bukan jaminan keamanan mutlak.",
        }

    def _fallback_file(self, hasil_file: Dict[str, Any], jumlah_url_berisiko: int = 0) -> Dict[str, str]:
        data = amankan_konteks_dict(hasil_file, KOLOM_FILE_PENTING)
        hasil = data.get("hasil_akhir_file_v3", "-")
        kategori = data.get("kategori_final_file_v3", "-")
        ekstensi = data.get("ekstensi", "-")
        jumlah_url = data.get("jumlah_url", "0")
        kata = data.get("kata_mencurigakan", "")
        rekomendasi_engine = data.get("rekomendasi_final_file_v3", "")

        alasan = [
            f"File berjenis {ekstensi}.",
            f"Sistem menemukan {jumlah_url} URL di dalam file.",
        ]

        if jumlah_url_berisiko:
            alasan.append(f"Ada {jumlah_url_berisiko} URL yang dinilai berisiko.")
        if kata:
            alasan.append(f"Ada kata yang perlu diwaspadai: {kata}.")

        if hasil == "Berisiko":
            tindakan = "Jangan buka file di perangkat utama. Periksa di lingkungan aman atau minta bantuan pihak yang memahami keamanan."
        elif hasil == "Perlu Tinjauan":
            tindakan = "Cek sumber file dan URL di dalamnya sebelum dibuka."
        else:
            tindakan = "File terlihat rendah risiko, tetapi tetap buka hanya jika sumbernya tepercaya."

        return {
            "mode": "fallback_lokal",
            "ringkasan": f"Hasil engine file: {hasil} dengan kategori {kategori}.",
            "alasan_sederhana": " ".join(alasan),
            "rekomendasi": rekomendasi_engine or tindakan,
            "batasan": "Sistem hanya membaca file secara statis dan tidak menjalankan isi file.",
        }

    def jelaskan_hasil_url(self, hasil_url: Dict[str, Any]) -> Dict[str, str]:
        """Menjelaskan hasil URL dari engine."""
        data = amankan_konteks_dict(hasil_url, KOLOM_URL_PENTING)

        prompt = f"""
{self._prompt_sistem()}

Buat jawaban dalam 4 bagian:
1. Ringkasan
2. Alasan sederhana
3. Rekomendasi tindakan
4. Batasan

Data hasil engine:
{json.dumps(data, ensure_ascii=False, indent=2)}
""".strip()

        jawaban_ai = self._panggil_ai(prompt)

        if jawaban_ai:
            return {
                "mode": "ai",
                "ringkasan": jawaban_ai,
                "alasan_sederhana": "",
                "rekomendasi": "",
                "batasan": "AI menjelaskan hasil engine. Keputusan utama tetap dari PhishRisk Engine.",
            }

        return self._fallback_url(hasil_url)

    def jelaskan_hasil_file(
        self,
        hasil_file: Dict[str, Any],
        hasil_url_dalam_file: Optional[pd.DataFrame] = None,
    ) -> Dict[str, str]:
        """Menjelaskan hasil file dari engine."""
        data_file = amankan_konteks_dict(hasil_file, KOLOM_FILE_PENTING)

        jumlah_url_berisiko = 0
        contoh_url = []

        if hasil_url_dalam_file is not None and not hasil_url_dalam_file.empty:
            if "hasil_akhir" in hasil_url_dalam_file.columns:
                jumlah_url_berisiko = int((hasil_url_dalam_file["hasil_akhir"] == "Berisiko").sum())

            if "url" in hasil_url_dalam_file.columns:
                contoh_url = potong_daftar_teks(hasil_url_dalam_file["url"].head(5).tolist(), batas_item=5)

        prompt = f"""
{self._prompt_sistem()}

Jelaskan hasil pemeriksaan file dalam 4 bagian:
1. Ringkasan
2. Alasan sederhana
3. Rekomendasi tindakan
4. Batasan

Data hasil file:
{json.dumps(data_file, ensure_ascii=False, indent=2)}

Jumlah URL berisiko di dalam file: {jumlah_url_berisiko}
Contoh URL yang ditemukan:
{json.dumps(contoh_url, ensure_ascii=False, indent=2)}
""".strip()

        jawaban_ai = self._panggil_ai(prompt)

        if jawaban_ai:
            return {
                "mode": "ai",
                "ringkasan": jawaban_ai,
                "alasan_sederhana": "",
                "rekomendasi": "",
                "batasan": "AI menjelaskan hasil engine. File tidak dijalankan.",
            }

        return self._fallback_file(hasil_file, jumlah_url_berisiko=jumlah_url_berisiko)

    def buat_ringkasan_batch(self, data_hasil: pd.DataFrame, jenis: str = "url") -> Dict[str, str]:
        """Meringkas banyak hasil URL atau file."""
        if data_hasil is None or data_hasil.empty:
            return {
                "mode": "fallback_lokal",
                "ringkasan": "Tidak ada data yang bisa diringkas.",
                "rekomendasi": "Jalankan pemeriksaan terlebih dahulu.",
            }

        total = len(data_hasil)
        kolom_hasil = "hasil_akhir" if jenis == "url" else "hasil_akhir_file_v3"
        kolom_kategori = "kategori_risiko" if jenis == "url" else "kategori_final_file_v3"

        hitung_hasil = data_hasil[kolom_hasil].value_counts().to_dict() if kolom_hasil in data_hasil.columns else {}
        hitung_kategori = data_hasil[kolom_kategori].value_counts().to_dict() if kolom_kategori in data_hasil.columns else {}

        konteks = {
            "jenis": jenis,
            "total_data": total,
            "ringkasan_hasil": hitung_hasil,
            "ringkasan_kategori": hitung_kategori,
        }

        prompt = f"""
{self._prompt_sistem()}

Buat ringkasan singkat dari hasil batch.
Gunakan bahasa sederhana.
Berikan kesimpulan dan tindakan yang disarankan.

Data ringkasan:
{json.dumps(konteks, ensure_ascii=False, indent=2)}
""".strip()

        jawaban_ai = self._panggil_ai(prompt)

        if jawaban_ai:
            return {
                "mode": "ai",
                "ringkasan": jawaban_ai,
                "rekomendasi": "Gunakan hasil detail untuk mengecek item paling berisiko.",
            }

        return {
            "mode": "fallback_lokal",
            "ringkasan": f"Total data diperiksa: {total}. Hasil: {hitung_hasil}. Kategori: {hitung_kategori}.",
            "rekomendasi": "Utamakan pemeriksaan pada item dengan kategori Tinggi atau Sangat Tinggi.",
        }

    def jawab_copilot(self, pertanyaan: str, konteks: Dict[str, Any] | None = None) -> Dict[str, str]:
        """Menjawab pertanyaan user berdasarkan konteks hasil PhishRisk."""
        berbahaya, alasan = deteksi_permintaan_berbahaya(pertanyaan)

        if berbahaya:
            return {
                "mode": "safety_guard",
                "jawaban": (
                    "Saya tidak bisa membantu permintaan yang mengarah ke penyalahgunaan. "
                    "Saya bisa membantu menjelaskan deteksi phishing, membuat rekomendasi aman, atau membuat laporan defensif."
                ),
                "alasan": alasan,
            }

        konteks_aman = konteks or {}

        prompt = f"""
{self._prompt_sistem()}

Jawab pertanyaan user berdasarkan konteks PhishRisk.
Jika informasi tidak cukup, jawab dengan jujur dan beri saran pemeriksaan aman.

Pertanyaan user:
{bersihkan_teks(pertanyaan, batas_karakter=1000)}

Konteks:
{json.dumps(konteks_aman, ensure_ascii=False, indent=2)}
""".strip()

        jawaban_ai = self._panggil_ai(prompt)

        if jawaban_ai:
            return {
                "mode": "ai",
                "jawaban": jawaban_ai,
                "alasan": "Jawaban dibuat berdasarkan konteks hasil PhishRisk.",
            }

        return {
            "mode": "fallback_lokal",
            "jawaban": (
                "Mode AI eksternal belum aktif. Berdasarkan konsep PhishRisk, gunakan hasil engine sebagai acuan utama, "
                "cek domain resmi, jangan klik link dari sumber asing, dan jangan membuka file mencurigakan di perangkat utama."
            ),
            "alasan": "Fallback lokal aktif karena API key tidak tersedia atau client AI gagal dipakai.",
        }'''

lokasi_ai_explainer = direktori_src / "ai_explainer.py"
lokasi_ai_explainer.write_text(isi_ai_explainer, encoding="utf-8")

print("Modul AI Explainer berhasil dibuat:")
print(lokasi_ai_explainer)


Modul AI Explainer berhasil dibuat:
C:\Users\ASUS\PHISHING\src\ai_explainer.py


## Membuat Modul AI Report Generator

In [5]:

isi_ai_report_generator = r'''from __future__ import annotations

from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional

import pandas as pd


def _hitung_kolom(data: pd.DataFrame, nama_kolom: str) -> Dict[str, int]:
    if data is None or data.empty or nama_kolom not in data.columns:
        return {}

    return data[nama_kolom].value_counts().to_dict()


def _format_dict(data: Dict[str, Any]) -> str:
    if not data:
        return "- Tidak ada data."

    baris = []
    for kunci, nilai in data.items():
        baris.append(f"- {kunci}: {nilai}")

    return "\n".join(baris)


def buat_laporan_url_markdown(
    data_url: pd.DataFrame,
    ringkasan_ai: Optional[Dict[str, str]] = None,
    judul: str = "Laporan Pemeriksaan URL PhishRisk",
) -> str:
    """Membuat laporan URL dalam format Markdown."""
    total = 0 if data_url is None else len(data_url)
    hasil = _hitung_kolom(data_url, "hasil_akhir")
    kategori = _hitung_kolom(data_url, "kategori_risiko")
    waktu = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    bagian_ai = ""
    if ringkasan_ai:
        bagian_ai = f"""
## Ringkasan Cerdas

{ringkasan_ai.get("ringkasan", "-")}

## Rekomendasi

{ringkasan_ai.get("rekomendasi", "-")}
""".strip()

    laporan = f"""
# {judul}

Waktu laporan: {waktu}

## Ringkasan Data

- Jumlah URL diperiksa: {total}

## Hasil Akhir

{_format_dict(hasil)}

## Kategori Risiko

{_format_dict(kategori)}

{bagian_ai}

## Catatan Keamanan

Hasil ini adalah bantuan awal. Jangan gunakan hasil model sebagai satu-satunya keputusan keamanan. 
Untuk URL yang masuk kategori Tinggi atau Sangat Tinggi, jangan login, jangan isi data pribadi, dan cek domain resmi dari sumber tepercaya.
""".strip()

    return laporan


def buat_laporan_file_markdown(
    data_file: pd.DataFrame,
    data_url_dalam_file: Optional[pd.DataFrame] = None,
    ringkasan_ai: Optional[Dict[str, str]] = None,
    judul: str = "Laporan Pemeriksaan File PhishRisk",
) -> str:
    """Membuat laporan file dalam format Markdown."""
    total_file = 0 if data_file is None else len(data_file)
    total_url = 0 if data_url_dalam_file is None else len(data_url_dalam_file)
    hasil_file = _hitung_kolom(data_file, "hasil_akhir_file_v3")
    kategori_file = _hitung_kolom(data_file, "kategori_final_file_v3")
    waktu = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    bagian_ai = ""
    if ringkasan_ai:
        bagian_ai = f"""
## Ringkasan Cerdas

{ringkasan_ai.get("ringkasan", "-")}

## Rekomendasi

{ringkasan_ai.get("rekomendasi", "-")}
""".strip()

    laporan = f"""
# {judul}

Waktu laporan: {waktu}

## Ringkasan Data

- Jumlah file diperiksa: {total_file}
- Jumlah URL ditemukan di dalam file: {total_url}

## Hasil File

{_format_dict(hasil_file)}

## Kategori Risiko File

{_format_dict(kategori_file)}

{bagian_ai}

## Catatan Keamanan

Pemeriksaan file dilakukan secara statis. Sistem tidak menjalankan file. 
Untuk file berisiko tinggi, jangan dibuka di perangkat utama dan jangan klik URL di dalamnya.
""".strip()

    return laporan


def simpan_laporan(teks_laporan: str, lokasi_output: str | Path) -> Path:
    """Menyimpan laporan ke file Markdown."""
    path_output = Path(lokasi_output)
    path_output.parent.mkdir(parents=True, exist_ok=True)
    path_output.write_text(teks_laporan, encoding="utf-8")
    return path_output'''

lokasi_ai_report_generator = direktori_src / "ai_report_generator.py"
lokasi_ai_report_generator.write_text(isi_ai_report_generator, encoding="utf-8")

print("Modul AI Report Generator berhasil dibuat:")
print(lokasi_ai_report_generator)


Modul AI Report Generator berhasil dibuat:
C:\Users\ASUS\PHISHING\src\ai_report_generator.py


## Membuat Modul AI Feedback Manager

In [6]:

isi_ai_feedback_manager = r'''from __future__ import annotations

from datetime import datetime
from pathlib import Path
from typing import Optional

import pandas as pd


KOLOM_FEEDBACK = [
    "waktu",
    "tipe",
    "input_user",
    "hasil_sistem",
    "kategori_risiko",
    "feedback_user",
    "catatan_user",
]


def siapkan_file_feedback(lokasi_feedback: str | Path) -> Path:
    """Menyiapkan file feedback user."""
    path_feedback = Path(lokasi_feedback)
    path_feedback.parent.mkdir(parents=True, exist_ok=True)

    if not path_feedback.exists():
        pd.DataFrame(columns=KOLOM_FEEDBACK).to_csv(path_feedback, index=False)

    return path_feedback


def simpan_feedback(
    lokasi_feedback: str | Path,
    tipe: str,
    input_user: str,
    hasil_sistem: str,
    kategori_risiko: str,
    feedback_user: str,
    catatan_user: Optional[str] = "",
) -> Path:
    """Menyimpan feedback user untuk koreksi dan evaluasi berikutnya."""
    path_feedback = siapkan_file_feedback(lokasi_feedback)

    data_lama = pd.read_csv(path_feedback)

    data_baru = pd.DataFrame(
        [
            {
                "waktu": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "tipe": tipe,
                "input_user": input_user,
                "hasil_sistem": hasil_sistem,
                "kategori_risiko": kategori_risiko,
                "feedback_user": feedback_user,
                "catatan_user": catatan_user or "",
            }
        ]
    )

    data_final = pd.concat([data_lama, data_baru], ignore_index=True)
    data_final.to_csv(path_feedback, index=False)

    return path_feedback


def baca_feedback(lokasi_feedback: str | Path) -> pd.DataFrame:
    """Membaca data feedback user."""
    path_feedback = siapkan_file_feedback(lokasi_feedback)
    return pd.read_csv(path_feedback)


def ringkas_feedback(lokasi_feedback: str | Path) -> pd.DataFrame:
    """Membuat ringkasan feedback user."""
    data = baca_feedback(lokasi_feedback)

    if data.empty:
        return pd.DataFrame(columns=["feedback_user", "jumlah_data"])

    return (
        data.groupby("feedback_user")
        .size()
        .reset_index(name="jumlah_data")
        .sort_values("jumlah_data", ascending=False)
    )'''

lokasi_ai_feedback_manager = direktori_src / "ai_feedback_manager.py"
lokasi_ai_feedback_manager.write_text(isi_ai_feedback_manager, encoding="utf-8")

print("Modul AI Feedback Manager berhasil dibuat:")
print(lokasi_ai_feedback_manager)


Modul AI Feedback Manager berhasil dibuat:
C:\Users\ASUS\PHISHING\src\ai_feedback_manager.py


## Membuat CLI AI Explainer

In [7]:

isi_run_ai_phishrisk = r'''from __future__ import annotations

import argparse
import sys
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ai_explainer import PhishRiskAIExplainer
from ai_report_generator import buat_laporan_url_markdown, simpan_laporan
from phishrisk_engine_v3 import PhishRiskEngineV3


def main() -> None:
    parser = argparse.ArgumentParser(description="PhishRisk AI Explainer CLI")
    parser.add_argument("--input", required=True, help="URL yang ingin diperiksa")
    parser.add_argument("--output", default="reports/outputs/laporan_ai_url.md", help="Lokasi output laporan Markdown")
    args = parser.parse_args()

    engine = PhishRiskEngineV3(direktori_project=ROOT)
    explainer = PhishRiskAIExplainer()

    hasil_url = engine.analisis_url(args.input)
    data_url = pd.DataFrame([hasil_url])
    ringkasan_ai = explainer.buat_ringkasan_batch(data_url, jenis="url")
    penjelasan = explainer.jelaskan_hasil_url(hasil_url)

    laporan = buat_laporan_url_markdown(data_url, ringkasan_ai=ringkasan_ai)
    laporan += "\n\n## Penjelasan URL\n\n"
    laporan += penjelasan.get("ringkasan", "")
    if penjelasan.get("alasan_sederhana"):
        laporan += "\n\n" + penjelasan.get("alasan_sederhana", "")
    if penjelasan.get("rekomendasi"):
        laporan += "\n\nRekomendasi: " + penjelasan.get("rekomendasi", "")

    output = ROOT / args.output
    simpan_laporan(laporan, output)

    print("Hasil AI Explainer")
    print("=" * 60)
    print("URL:", args.input)
    print("Hasil:", hasil_url.get("hasil_akhir"))
    print("Kategori:", hasil_url.get("kategori_risiko"))
    print("Laporan:", output)


if __name__ == "__main__":
    main()'''

lokasi_run_ai_phishrisk = direktori_src / "run_ai_phishrisk.py"
lokasi_run_ai_phishrisk.write_text(isi_run_ai_phishrisk, encoding="utf-8")

print("CLI AI Explainer berhasil dibuat:")
print(lokasi_run_ai_phishrisk)


CLI AI Explainer berhasil dibuat:
C:\Users\ASUS\PHISHING\src\run_ai_phishrisk.py


## Import Modul dan Cek Mode AI

Jika belum punya API key, sistem akan memakai fallback lokal. Ini aman untuk testing awal.

In [8]:

from ai_explainer import PhishRiskAIExplainer
from ai_report_generator import buat_laporan_url_markdown, buat_laporan_file_markdown, simpan_laporan
from ai_feedback_manager import simpan_feedback, baca_feedback, ringkas_feedback
from phishrisk_engine_v3 import PhishRiskEngineV3

engine = PhishRiskEngineV3(direktori_project=direktori_project)
explainer = PhishRiskAIExplainer()

print("Mode AI aktif:", explainer.aktifkan_ai)
print("Model AI:", explainer.model_ai)
print("Catatan: jika False, sistem tetap berjalan memakai fallback lokal.")


Mode AI aktif: False
Model AI: gpt-5.5-mini
Catatan: jika False, sistem tetap berjalan memakai fallback lokal.


## Membuat Contoh URL untuk Uji AI

In [9]:

daftar_url_uji_ai = [
    "https://praktikum.gunadarma.ac.id",
    "https://baak.gunadarma.ac.id",
    "https://www.bca.co.id",
    "https://www.shopee.co.id",
    "https://www.microsoft.com",
    "http://rricrosoft.com",
    "http://rnicrosoft.com",
    "http://micros0ft-login-update.test",
    "http://bca-login-update.test",
    "http://paypal-verify-account.test",
    "https://xn--micrsoft-q4a.test",
]

data_url_uji_ai = pd.DataFrame({"url": daftar_url_uji_ai})
lokasi_contoh_ai = direktori_examples / "input_url_step11_ai.csv"
data_url_uji_ai.to_csv(lokasi_contoh_ai, index=False)

print("Contoh input URL AI berhasil dibuat:")
print(lokasi_contoh_ai)
display(data_url_uji_ai)


Contoh input URL AI berhasil dibuat:
C:\Users\ASUS\PHISHING\examples\input_url_step11_ai.csv


,url
0,https://praktikum.gunadarma.ac.id
1,https://baak.gunadarma.ac.id
2,https://www.bca.co.id
3,https://www.shopee.co.id
4,https://www.microsoft.com
5,http://rricrosoft.com
6,http://rnicrosoft.com
7,http://micros0ft-login-update.test
8,http://bca-login-update.test
9,http://paypal-verify-account.test


## Uji Engine V3 + AI Explainer

In [10]:

hasil_url_ai = []

for url in daftar_url_uji_ai:
    hasil = engine.analisis_url(url)
    penjelasan = explainer.jelaskan_hasil_url(hasil)

    hasil_ringkas = {
        "url": hasil.get("url"),
        "domain": hasil.get("domain"),
        "hasil_akhir": hasil.get("hasil_akhir"),
        "kategori_risiko": hasil.get("kategori_risiko"),
        "skor_final": hasil.get("skor_final"),
        "intelligence_status": hasil.get("intelligence_status"),
        "mode_penjelasan": penjelasan.get("mode"),
        "ringkasan_ai": penjelasan.get("ringkasan"),
        "alasan_sederhana": penjelasan.get("alasan_sederhana"),
        "rekomendasi_ai": penjelasan.get("rekomendasi"),
    }

    hasil_url_ai.append(hasil_ringkas)

data_hasil_url_ai = pd.DataFrame(hasil_url_ai)
lokasi_hasil_url_ai = direktori_outputs / "hasil_ai_explainer_url_step11.csv"
data_hasil_url_ai.to_csv(lokasi_hasil_url_ai, index=False)

print("Hasil AI Explainer URL disimpan:")
print(lokasi_hasil_url_ai)
display(data_hasil_url_ai)


Hasil AI Explainer URL disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_ai_explainer_url_step11.csv


,url,domain,hasil_akhir,kategori_risiko,skor_final,intelligence_status,mode_penjelasan,ringkasan_ai,alasan_sederhana,rekomendasi_ai
0,https://praktikum.gunadarma.ac.id,praktikum.gunadarma.ac.id,Terlihat Aman,Rendah,20.40,resmi_terlihat_aman,fallback_lokal,Hasil engine: Terlihat Aman dengan kategori Re...,Domain cocok dengan daftar pembanding resmi. S...,Alamat cocok dengan daftar domain resmi dan ti...
1,https://baak.gunadarma.ac.id,baak.gunadarma.ac.id,Terlihat Aman,Rendah,24.00,resmi_terlihat_aman,fallback_lokal,Hasil engine: Terlihat Aman dengan kategori Re...,Domain cocok dengan daftar pembanding resmi. S...,Alamat cocok dengan daftar domain resmi dan ti...
2,https://www.bca.co.id,www.bca.co.id,Terlihat Aman,Rendah,4.05,resmi_terlihat_aman,fallback_lokal,Hasil engine: Terlihat Aman dengan kategori Re...,Domain cocok dengan daftar pembanding resmi. S...,Alamat cocok dengan daftar domain resmi dan ti...
3,https://www.shopee.co.id,www.shopee.co.id,Terlihat Aman,Rendah,9.22,resmi_terlihat_aman,fallback_lokal,Hasil engine: Terlihat Aman dengan kategori Re...,Domain cocok dengan daftar pembanding resmi. S...,Alamat cocok dengan daftar domain resmi dan ti...
4,https://www.microsoft.com,www.microsoft.com,Terlihat Aman,Rendah,24.00,resmi_terlihat_aman,fallback_lokal,Hasil engine: Terlihat Aman dengan kategori Re...,Domain cocok dengan daftar pembanding resmi. S...,Alamat cocok dengan daftar domain resmi dan ti...
5,http://rricrosoft.com,rricrosoft.com,Berisiko,Sangat Tinggi,99.60,domain_mirip_brand,fallback_lokal,Hasil engine: Berisiko dengan kategori Sangat ...,Domain terlihat mirip dengan brand resmi. Skor...,Alamat terindikasi meniru brand atau domain re...
6,http://rnicrosoft.com,rnicrosoft.com,Berisiko,Sangat Tinggi,99.60,domain_mirip_brand,fallback_lokal,Hasil engine: Berisiko dengan kategori Sangat ...,Domain terlihat mirip dengan brand resmi. Skor...,Alamat terindikasi meniru brand atau domain re...
7,http://micros0ft-login-update.test,micros0ft-login-update.test,Berisiko,Sangat Tinggi,98.80,tiruan_brand_berisiko,fallback_lokal,Hasil engine: Berisiko dengan kategori Sangat ...,"Alamat terlihat memakai nama brand, tetapi buk...",Alamat terindikasi meniru brand atau domain re...
8,http://bca-login-update.test,bca-login-update.test,Berisiko,Sangat Tinggi,99.60,tiruan_brand_berisiko,fallback_lokal,Hasil engine: Berisiko dengan kategori Sangat ...,"Alamat terlihat memakai nama brand, tetapi buk...",Alamat terindikasi meniru brand atau domain re...
9,http://paypal-verify-account.test,paypal-verify-account.test,Berisiko,Sangat Tinggi,100.00,tiruan_brand_berisiko,fallback_lokal,Hasil engine: Berisiko dengan kategori Sangat ...,"Alamat terlihat memakai nama brand, tetapi buk...",Alamat terindikasi meniru brand atau domain re...


## Ringkasan Cerdas Batch URL

In [11]:

data_engine_url = pd.DataFrame([engine.analisis_url(url) for url in daftar_url_uji_ai])
ringkasan_batch_url = explainer.buat_ringkasan_batch(data_engine_url, jenis="url")

print("Ringkasan batch URL:")
for kunci, nilai in ringkasan_batch_url.items():
    print(f"{kunci}: {nilai}")


Ringkasan batch URL:
mode: fallback_lokal
ringkasan: Total data diperiksa: 11. Hasil: {'Berisiko': 6, 'Terlihat Aman': 5}. Kategori: {'Sangat Tinggi': 6, 'Rendah': 5}.
rekomendasi: Utamakan pemeriksaan pada item dengan kategori Tinggi atau Sangat Tinggi.


## Membuat Laporan Markdown URL

In [12]:

laporan_url = buat_laporan_url_markdown(
    data_engine_url,
    ringkasan_ai=ringkasan_batch_url,
    judul="Laporan AI Pemeriksaan URL PhishRisk",
)

lokasi_laporan_url = direktori_outputs / "laporan_ai_pemeriksaan_url_step11.md"
simpan_laporan(laporan_url, lokasi_laporan_url)

print("Laporan URL berhasil disimpan:")
print(lokasi_laporan_url)
print("\nPreview laporan:")
print(laporan_url[:1500])


Laporan URL berhasil disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\laporan_ai_pemeriksaan_url_step11.md

Preview laporan:
# Laporan AI Pemeriksaan URL PhishRisk

Waktu laporan: 2026-05-16 13:59:46

## Ringkasan Data

- Jumlah URL diperiksa: 11

## Hasil Akhir

- Berisiko: 6
- Terlihat Aman: 5

## Kategori Risiko

- Sangat Tinggi: 6
- Rendah: 5

## Ringkasan Cerdas

Total data diperiksa: 11. Hasil: {'Berisiko': 6, 'Terlihat Aman': 5}. Kategori: {'Sangat Tinggi': 6, 'Rendah': 5}.

## Rekomendasi

Utamakan pemeriksaan pada item dengan kategori Tinggi atau Sangat Tinggi.

## Catatan Keamanan

Hasil ini adalah bantuan awal. Jangan gunakan hasil model sebagai satu-satunya keputusan keamanan. 
Untuk URL yang masuk kategori Tinggi atau Sangat Tinggi, jangan login, jangan isi data pribadi, dan cek domain resmi dari sumber tepercaya.


## Uji AI Feedback Manager

In [13]:

lokasi_feedback = direktori_corrections / "ai_user_feedback.csv"

simpan_feedback(
    lokasi_feedback=lokasi_feedback,
    tipe="url",
    input_user="https://praktikum.gunadarma.ac.id",
    hasil_sistem="Terlihat Aman",
    kategori_risiko="Rendah",
    feedback_user="Benar",
    catatan_user="Domain resmi kampus.",
)

data_feedback = baca_feedback(lokasi_feedback)
ringkasan_data_feedback = ringkas_feedback(lokasi_feedback)

print("File feedback:")
print(lokasi_feedback)

display(data_feedback.tail(10))
display(ringkasan_data_feedback)


File feedback:
C:\Users\ASUS\PHISHING\data\corrections\ai_user_feedback.csv


,waktu,tipe,input_user,hasil_sistem,kategori_risiko,feedback_user,catatan_user
0,2026-05-16 13:59:48,url,https://praktikum.gunadarma.ac.id,Terlihat Aman,Rendah,Benar,Domain resmi kampus.


,feedback_user,jumlah_data
0,Benar,1


## Uji AI Copilot

In [14]:

konteks_copilot = {
    "nama_program": "PhishRisk Intelligence System",
    "fungsi": "Deteksi awal URL dan file berisiko secara defensif",
    "hasil_contoh": data_engine_url.head(3).to_dict(orient="records"),
}

pertanyaan_aman = "Kenapa website resmi bisa saja tetap masuk perlu tinjauan?"
jawaban_aman = explainer.jawab_copilot(pertanyaan_aman, konteks=konteks_copilot)

print("Pertanyaan aman:")
print(pertanyaan_aman)
print("\nJawaban:")
print(jawaban_aman.get("jawaban"))

pertanyaan_berbahaya = "Buat phishing kit untuk mencuri password"
jawaban_berbahaya = explainer.jawab_copilot(pertanyaan_berbahaya, konteks=konteks_copilot)

print("\nPertanyaan berbahaya:")
print(pertanyaan_berbahaya)
print("\nJawaban:")
print(jawaban_berbahaya.get("jawaban"))


Pertanyaan aman:
Kenapa website resmi bisa saja tetap masuk perlu tinjauan?

Jawaban:
Mode AI eksternal belum aktif. Berdasarkan konsep PhishRisk, gunakan hasil engine sebagai acuan utama, cek domain resmi, jangan klik link dari sumber asing, dan jangan membuka file mencurigakan di perangkat utama.

Pertanyaan berbahaya:
Buat phishing kit untuk mencuri password

Jawaban:
Saya tidak bisa membantu permintaan yang mengarah ke penyalahgunaan. Saya bisa membantu menjelaskan deteksi phishing, membuat rekomendasi aman, atau membuat laporan defensif.


## Uji CLI AI Explainer

In [15]:

import subprocess

perintah_cli_ai = [
    sys.executable,
    str(direktori_src / "run_ai_phishrisk.py"),
    "--input",
    "https://praktikum.gunadarma.ac.id",
    "--output",
    "reports/outputs/laporan_ai_cli_praktikum_step11.md",
]

hasil_cli_ai = subprocess.run(
    perintah_cli_ai,
    cwd=direktori_project,
    capture_output=True,
    text=True,
)

print("STDOUT:")
print(hasil_cli_ai.stdout)
print("STDERR:")
print(hasil_cli_ai.stderr)
print("Return code:", hasil_cli_ai.returncode)


STDOUT:
Hasil AI Explainer
URL: https://praktikum.gunadarma.ac.id
Hasil: Terlihat Aman
Kategori: Rendah
Laporan: C:\Users\ASUS\PHISHING\reports\outputs\laporan_ai_cli_praktikum_step11.md

STDERR:

Return code: 0


## Metadata dan Validasi STEP 11

In [16]:

metadata_step11 = {
    "nama_program": "PhishRisk AI Explainer Layer",
    "step": "STEP 11",
    "status": "AI Explainer Layer selesai",
    "direktori_project": str(direktori_project),
    "mode_ai_aktif": explainer.aktifkan_ai,
    "model_ai": explainer.model_ai,
    "komponen_baru": [
        "ai_safety_guard",
        "ai_explainer",
        "ai_report_generator",
        "ai_feedback_manager",
        "run_ai_phishrisk",
    ],
    "file_baru": {
        "ai_safety_guard": str(lokasi_ai_safety_guard),
        "ai_explainer": str(lokasi_ai_explainer),
        "ai_report_generator": str(lokasi_ai_report_generator),
        "ai_feedback_manager": str(lokasi_ai_feedback_manager),
        "run_ai_phishrisk": str(lokasi_run_ai_phishrisk),
        "hasil_ai_url": str(lokasi_hasil_url_ai),
        "laporan_ai_url": str(lokasi_laporan_url),
        "feedback": str(lokasi_feedback),
    },
    "catatan": "AI menjelaskan hasil engine dan tidak mengganti keputusan utama PhishRisk Engine.",
}

lokasi_metadata_step11 = direktori_outputs / "metadata_step11_ai_explainer.json"
lokasi_metadata_step11.write_text(json.dumps(metadata_step11, indent=2, ensure_ascii=False), encoding="utf-8")

file_validasi_step11 = {
    "ai_safety_guard.py": lokasi_ai_safety_guard,
    "ai_explainer.py": lokasi_ai_explainer,
    "ai_report_generator.py": lokasi_ai_report_generator,
    "ai_feedback_manager.py": lokasi_ai_feedback_manager,
    "run_ai_phishrisk.py": lokasi_run_ai_phishrisk,
    "input_url_step11_ai.csv": lokasi_contoh_ai,
    "hasil_ai_explainer_url_step11.csv": lokasi_hasil_url_ai,
    "laporan_ai_pemeriksaan_url_step11.md": lokasi_laporan_url,
    "ai_user_feedback.csv": lokasi_feedback,
    "metadata_step11_ai_explainer.json": lokasi_metadata_step11,
}

data_validasi_step11 = []

for nama_file, lokasi in file_validasi_step11.items():
    data_validasi_step11.append(
        {
            "nama_file": nama_file,
            "lokasi": str(lokasi),
            "tersedia": lokasi.exists(),
            "ukuran_kb": round(lokasi.stat().st_size / 1024, 2) if lokasi.exists() else 0,
        }
    )

data_validasi_step11 = pd.DataFrame(data_validasi_step11)
lokasi_validasi_step11 = direktori_outputs / "validasi_step11_ai_explainer.csv"
data_validasi_step11.to_csv(lokasi_validasi_step11, index=False)

print("Metadata STEP 11 disimpan:")
print(lokasi_metadata_step11)
print("\nValidasi STEP 11 disimpan:")
print(lokasi_validasi_step11)

display(data_validasi_step11)


Metadata STEP 11 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\metadata_step11_ai_explainer.json

Validasi STEP 11 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\validasi_step11_ai_explainer.csv


,nama_file,lokasi,tersedia,ukuran_kb
0,ai_safety_guard.py,C:\Users\ASUS\PHISHING\src\ai_safety_guard.py,True,2.68
1,ai_explainer.py,C:\Users\ASUS\PHISHING\src\ai_explainer.py,True,12.55
2,ai_report_generator.py,C:\Users\ASUS\PHISHING\src\ai_report_generator.py,True,3.40
3,ai_feedback_manager.py,C:\Users\ASUS\PHISHING\src\ai_feedback_manager.py,True,2.25
4,run_ai_phishrisk.py,C:\Users\ASUS\PHISHING\src\run_ai_phishrisk.py,True,1.79
5,input_url_step11_ai.csv,C:\Users\ASUS\PHISHING\examples\input_url_step...,True,0.32
6,hasil_ai_explainer_url_step11.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_a...,True,4.39
7,laporan_ai_pemeriksaan_url_step11.md,C:\Users\ASUS\PHISHING\reports\outputs\laporan...,True,0.72
8,ai_user_feedback.csv,C:\Users\ASUS\PHISHING\data\corrections\ai_use...,True,0.18
9,metadata_step11_ai_explainer.json,C:\Users\ASUS\PHISHING\reports\outputs\metadat...,True,1.18
